# Fase de razonamiento previo

El sistema actual utiliza una única regla para activar un nuevo modelo: `accuracy_nuevo >= accuracy_anterior`.

Sin embargo, en entornos reales no siempre es suficiente utilizar una única métrica global como la accuracy para decidir si un modelo debe reemplazar al anterior. Dependiendo del contexto del problema, pueden ser necesarias políticas de activación distintas.

A continuación, analizo los tres escenarios propuestos y justifico qué política aplicaría en cada uno.

## Escenario 1: Detección de fraude bancario

En este caso, los falsos negativos (no detectar un fraude) son mucho más costosos que los falsos positivos.

Aquí la accuracy global no es suficiente, ya que un modelo puede tener buena accuracy pero seguir fallando en la clase más crítica (fraude). Lo importante es mejorar la capacidad de detectar correctamente los casos de fraude.

Por tanto, aplicaría la política: `per_class_f1`. El modelo solo se activaría si mejora el F1-score de la clase “fraude”.

El F1-score es adecuado porque combina precisión y recall, y en este contexto el recall es especialmente importante para evitar que se escapen casos fraudulentos.

En este escenario, priorizamos el rendimiento en una clase específica sobre la métrica global.

## Escenario 2: Sistema de recomendación de películas

En este caso, el coste del error es bajo. Una recomendación ligeramente peor no genera un impacto grave.

Aquí interesa evolucionar rápidamente y aceptar mejoras aunque sean pequeñas.

Aplicaría la política: `any_improvement`. El modelo se activa si la accuracy nueva es mayor o igual que la anterior.

En este tipo de sistemas es razonable aceptar mejoras marginales, ya que el riesgo asociado es reducido y se favorece la iteración continua.

## Escenario 3: Modelo médico de diagnóstico

En este contexto, la estabilidad y la previsibilidad son fundamentales. No se debe cambiar un modelo en producción por mejoras mínimas que podrían deberse al ruido del dataset.

Aquí aplicaría la política: `min_delta`. El modelo solo se activaría si mejora la accuracy al menos en un porcentaje configurable (por ejemplo, un 2%).

De este modo, evitamos activar modelos cuya mejora no sea significativa. Esto aporta mayor estabilidad al sistema y reduce cambios frecuentes sin impacto real.

## Diseño de la clase ActivationPolicy

Para implementar estas políticas de forma limpia y extensible, diseñaría una clase ActivationPolicy que encapsule la lógica de decisión.

Esta clase recibiría:

- accuracy_new
    
- accuracy_prev
    
- métricas adicionales (por ejemplo F1 por clase)
    
- el modo de política seleccionado

- parámetros adicionales según la política (por ejemplo min_delta o target_class)

Y devolvería:

- activate (True/False)

- reason (mensaje explicativo para registrar en el historial)

## Parámetros necesarios por política

`any_improvement` No requiere parámetros adicionales.

`min_delta`:

- min_delta: porcentaje mínimo de mejora exigido (por ejemplo 0.02).

`per_class_f1`:

- target_class: clase sobre la que se exige mejora.

- Cálculo de F1-score por clase.